# Adding more R libraries

The process for adding new R packages involves a few steps

## Step 1

Use the [R Documentation Search](https://www.rdocumentation.org/) to find the functionality you want, taking care to note what **library** it comes from.

For example the `anova()` function [comes from](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/anova) `library('stats')` which is typically autoloaded for you when you use R/RStudio.


## Step 2

Use `importr` from `rpy2` to load the package in Python and inspect what the **Python converted name** of the that function is. The easiest way to do this is to just use your code editor try to auto-complete after typing a `.` to see what functions are available.

Sometimes, the `._exported_names` attribute of the R library will contain a list of functions available. In this case, we need the `broom.tidy_lm` function which *isn't* listed in the `._exported_names`, but does get auto-completed by our code editor:


In [ ]:
from rpy2.robjects.packages import importr

broom = importr('broom')

# This is the function we need
help(broom.tidy_lm)

## Step 3

Try out the function to determine its input and output types. Below we use the `lm()` function already implemented in tidystats to create a model. In R, `tidy()` takes this model as input. So let's see what type it has:

In [1]:
from tidystats import lm
import polars as pl

df = pl.DataFrame({'x': [1,2,3,4,5], 'y':[10,30,20,50,40]})

model = lm('y ~ x', data=df)

type(model)

rpy2.robjects.vectors.ListVector

Then let's try out our function and determine the **output type**:

In [9]:
tidy_summary = broom.tidy_lm(model)
type(tidy_summary)

rpy2.robjects.vectors.DataFrame

## Step 4

Now that we know the input and output types we're dealing with we can create our own `tidy()` function automatically converts inputs and outputs properly between R and Python. 

For example, we don't want to work with a `rpy2.robjects.vectors.DataFrame`, but rather a polars DataFrame in Python. We can do this by using any number of **function decorators** in `tidystats.bridge`. These get evaluated from **bottom-to-top** and handle all kinds of things like ensurin `lm()` receives an R dataframe instead of a polars dataframe without you having to worry about it.

Create a new module, a new `.py` file if there doesn't already exist one for the library. In this case we'll create `broom.py`

Inside the file we can create our Python function that wraps the R functionality we want and gracefully handles any additional conversion between Python and R. In the example below we use the `@ensure_py_output` to ensure the result of our `tidy()` function is a polars DataFrame. And we use `@sanitize_polars_columns` to ensure that column names use `_` convention in Python instead of `.` like in R.

In [13]:
# In broom.py
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from tidystats.bridge import ensure_py_output, sanitize_polars_columns

# Import library
broom = importr("broom")

# Write our function
@sanitize_polars_columns # will rename columns nicely
@ensure_py_output # will auto-convert to polars df
def tidy(model):
    # Make sure we're working with an lm() model type
    if isinstance(model, ro.vectors.ListVector):
        return broom.tidy_lm(model)


Let's try our function out with the same model as before:

In [14]:
tidy(model)

term,estimate,std_error,statistic,p_value
str,f64,f64,f64,f64
"""(Intercept)""",6.0,11.489125,0.522233,0.637618
"""x""",8.0,3.464102,2.309401,0.104088


And we get back a nicely useable polars DataFrame, with all the calculations happening in R!

## Step 5

Open a pull-request on github to contribute the new functionality you added back to `pymer4` by checking out the [Contribution Guide](../../contributing/)